# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/siddanger7/flyrank-ml-internship-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

Lane 4: **CTR / Engagement Opportunity Scoring**. This notebook maps my lane onto the ML loop: task type, target/proxy, success metric, the unit of analysis as a real dataframe, and why ML beats a fixed rule here.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/siddanger7/flyrank-ml-internship-"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Lane slice: visible pages with a real position (avg_position == 0 means 'no data').
vis = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()

print("Working dir:", os.getcwd())
print("Starter rows:", len(df), "| lane slice (visible, has position):", len(vis))

Working dir: C:\Users\sidda\AppData\Local\Temp\opencode\repo
Starter rows: 30000 | lane slice (visible, has position): 22006


## 1. My lane as an ML task (type)

**Task type: scoring / ranking.**

My lane asks *"which visible pages should a content editor review first for CTR / title-meta improvement?"* — that is a *"which ones first?"* problem, so the output is a **ranked priority score**, not a cluster and not a yes/no classification on its own.

- Not classification: I am not predicting "declining or not"; I am ordering candidates by how far below their position tier's expectation they sit.
- Not clustering: there are no archetypes to discover — I already know the unit is a page and the goal is a review order.
- Scoring/ranking: each visible page gets a score, the list is ordered, and a reviewer works down it. The metric must reward the *top of the list* (Precision@K), because review capacity is small.

The critical constraint that makes this a *position-adjusted* scoring problem: CTR only means something **within** a position tier (page 1 always clicks better than page 9). So the score is "how much this page under-captures **for where it ranks**," never raw CTR alone.

In [2]:
# The ranking surface: how the slice splits across position tiers.
print("Position-tier mix of the lane slice:")
print(vis["position_tier"].value_counts().to_string())
print("\n=> A scoring/ranking task, one score per page, compared WITHIN a tier.")

Position-tier mix of the lane slice:
position_tier
page_1      8633
page_3_5    6058
striking    5903
deep         879
top_3        533

=> A scoring/ranking task, one score per page, compared WITHIN a tier.


## 2. Target or proxy

**The thing I rank by: a derived under-capture score.** For each visible page, compare its actual CTR to its position tier's benchmark:

- `under_gap` = tier median CTR − page CTR  (positive = below its tier's typical CTR)
- a **proxy binary label** `under_capturing` = CTR below half the tier median **and** enough volume (`impressions_90d >= 250`) so low-volume noise doesn't drive recommendations.

**Where the label comes from — honesty check.** This proxy is a **derived measurement from observed current signals** (CTR, position tier, impressions), not a product decision flag and not a future outcome. That is acceptable for framing the task, but I note the upgrade path: the strongest capstone label would be a **future outcome** measured in a later window (e.g., "did CTR improve 30 days after a title/meta review?") using the warehouse daily table (Week 3+). The starter data is a trailing-90-day snapshot, so a future-window label is not possible here — I say that out loud rather than pretend otherwise.

Guardrails kept: `trend_direction` and `trend_pct` are never features (they encode the label); `avg_position == 0` rows are filtered as "no data", not treated as rank zero; `ctr` is a ×100 rate (0.76 = 0.76%).

In [3]:
# Build the derived score + proxy label from observed signals.
tier_median = vis.groupby("position_tier")["ctr"].transform("median")
vis["under_gap"] = (tier_median - vis["ctr"]).round(3)
vis["under_ratio"] = np.where(tier_median > 0, (vis["ctr"] / tier_median).round(2), np.nan)
vis["under_capturing"] = ((vis["ctr"] < 0.5 * tier_median) & (vis["impressions_90d"] >= 250)).astype(int)

print("Proxy label 'under_capturing' rate in the slice:", round(vis["under_capturing"].mean(), 3))
print("Count:", int(vis["under_capturing"].sum()), "of", len(vis))
print("\nSample (one row = one page):")
print(vis[["position_tier", "ctr", "under_gap", "under_ratio", "under_capturing"]].head(8).to_string(index=False))

Proxy label 'under_capturing' rate in the slice: 0.253
Count: 5564 of 22006

Sample (one row = one page):
position_tier  ctr  under_gap  under_ratio  under_capturing
     striking 0.76      -0.61         5.07                0
     page_3_5 0.05       0.01         0.83                0
     page_3_5 0.09      -0.03         1.50                0
       page_1 0.49      -0.26         2.13                0
     page_3_5 0.13      -0.07         2.17                0
       page_1 0.03       0.20         0.13                1
     page_3_5 0.06       0.00         1.00                0
     page_3_5 0.09      -0.03         1.50                0


## 3. Success metric

**Precision@K on the ranked review queue** — the fraction of the top-K pages a ranking flags that are actually under-capturing (by the proxy label). This matches the real decision: an editor reviews a small number of pages per sprint, so only the top of the list matters.

- **Precision@20** if a reviewer checks 20 pages; **Precision@50** for a bigger sprint. I report both.
- Why not accuracy: the label is imbalanced (~20-30% positive) and a classifier that just says "no" everywhere looks "accurate" but is useless for review. Top-K precision rewards getting the *right handful* right.

To be honest, I must compare against a **transparent baseline rule**, not just quote a score. Here: the naive *"rank by raw CTR, lowest first"* rule versus the position-adjusted *"rank by under-gap"* score. If the adjusted score doesn't beat the naive rule at the top of the list, the framing isn't earning its keep.

In [4]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = vis["under_capturing"].values

# Baseline hand rule: raw CTR, lowest first ('fix the low-CTR page').
raw_ctr_score = -vis["ctr"].values
# Position-adjusted: rank by how far below the tier benchmark the page sits.
gap_score = vis["under_gap"].fillna(0).values

for k in (20, 50):
    b = precision_at_k(raw_ctr_score, y, k)
    g = precision_at_k(gap_score, y, k)
    print(f"Precision@{k}:  raw-CTR rule {b:.3f}   vs   tier-gap score {g:.3f}")

Precision@20:  raw-CTR rule 0.700   vs   tier-gap score 0.800
Precision@50:  raw-CTR rule 0.560   vs   tier-gap score 0.680


## 4. The unit of analysis, as a real dataframe

**One row = one content page** (grain: `content_id`), restricted to my lane slice: pages with real impressions (`impressions_90d >= 100`) and a real position (`avg_position > 0`).

The dataframe below is the literal unit of analysis for the whole lane — each row is a page that a reviewer could rank, with the signals that feed the score and the derived proxy label attached.

In [5]:
print("Grain: one row = one content page.  Rows:", vis.shape[0],
      "| unique content_id:", vis["content_id"].nunique())
show = vis[["content_id", "client_id", "content_type", "position_tier",
            "avg_position", "impressions_90d", "ctr", "under_gap", "under_capturing"]].head(6)
show

Grain: one row = one content page.  Rows: 22006 | unique content_id: 22006


,content_id,client_id,content_type,position_tier,avg_position,impressions_90d,ctr,under_gap,under_capturing
0,content_304f48230142,client_f369cb89fc,keyword article,striking,10.6,3803,0.76,-0.61,0
1,content_a1fb4e703a9e,client_4e07408562,keyword article,page_3_5,20.3,15320,0.05,0.01,0
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,page_3_5,36.5,12581,0.09,-0.03,0
3,content_331d6c4de07b,client_19581e27de,keyword article,page_1,6.2,11751,0.49,-0.26,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,page_3_5,44.0,19140,0.13,-0.07,0
5,content_d4084a4bc775,client_f369cb89fc,keyword article,page_1,8.5,3970,0.03,0.20,1


## 5. Why ML beats a fixed rule here

A fixed rule like *"CTR below 0.10 is a problem"* fails because **raw CTR is dominated by position**: a deep page at CTR 0.05 might be *ahead of its tier* while a page-1 page at CTR 0.15 is *far behind its tier*. The same number means opposite things depending on where the page ranks.

The pattern is too messy for an if-statement because:
- **Position interaction** — the meaning of a CTR number changes with the tier (and with avg_position continuously, not just the coarse tier).
- **Volume noise** — low-impression pages have unstable CTR; the rule must know when there is enough data to speak at all.
- **Many interacting signals** — content type, intent, age/freshness, engagement, and AI traffic all shift what "under-capturing" looks like; no single threshold captures the combination.

So the learned model's job is to learn *"weak for where it ranks, at scale, with enough volume"* from the feature set — then the output is still a ranked list a human reviews (decision support, not automation). The code below shows why the naive rule mis-ranks: checking what the top-50 of each rule actually looks like by position tier.

In [6]:
# Why a fixed 'low CTR' rule fails: CTR tracks position.
corr = vis["avg_position"].corr(vis["ctr"])
print(f"Correlation(avg_position, ctr) = {corr:.3f}  (lower avg_position = higher rank)")

def topk_tier_mix(scores, k=50):
    return vis["position_tier"].iloc[np.argsort(-np.asarray(scores))[:k]]\
        .value_counts(normalize=True).round(2)

print("\nTier mix of the top-50 by the raw-CTR rule (lowest CTR first):")
print(topk_tier_mix(raw_ctr_score).to_string())
print("\nTier mix of the top-50 by the position-adjusted under-gap score:")
print(topk_tier_mix(gap_score).to_string())
print("\nThe raw-CTR rule spends its top-50 on the lowest absolute CTRs (deep + page_3_5);"
      "the position-adjusted gap score spends it on page-1 pages sitting below their tier's"
      "benchmark. Same pages, very different orders — the pattern a model learns.")

Correlation(avg_position, ctr) = -0.239  (lower avg_position = higher rank)

Tier mix of the top-50 by the raw-CTR rule (lowest CTR first):
position_tier
page_3_5    0.36
page_1      0.26
striking    0.24
deep        0.12
top_3       0.02

Tier mix of the top-50 by the position-adjusted under-gap score:
position_tier
page_1    1.0

The raw-CTR rule spends its top-50 on the lowest absolute CTRs (deep + page_3_5);the position-adjusted gap score spends it on page-1 pages sitting below their tier'sbenchmark. Same pages, very different orders — the pattern a model learns.


## Self-check

Before submitting, each line is confirmed:

- [x] Task type named: scoring / ranking (which pages first)
- [x] Target/proxy defined: under-gap score + `under_capturing` proxy label, from observed signals
- [x] Success metric defended: Precision@K, vs a transparent baseline rule
- [x] Unit of analysis shown as a real dataframe: one row = one content page
- [x] Why ML beats a fixed rule explained: position interaction, volume noise, many signals
- [x] Output ties to a real action: a ranked review queue for a content editor
- [x] Runs top to bottom, no errors; careful observed/directional wording; no private data